# **Exercícios Aula 07**: Introdução ao Machine Learning

----

- **Dataset**: `sample_training` (MongoDB Atlas) — coleção **`trips`** (CitiBike NYC)
- **Objetivo**: Aplicar o pipeline completo de ML supervisionado com feature selection baseada em correlação.

-----------------

In [0]:
!pip install pymongo seaborn matplotlib scipy

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests, json
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# --- Bloco de preparação compartilhado por todos os exercícios ---
# O dataset sample_training não está carregado no MongoDB Atlas,
# então baixamos os dados públicos do GitHub (mesmo dataset oficial MongoDB).
_url = "https://raw.githubusercontent.com/neelabalan/mongodb-sample-dataset/main/sample_training/trips.json"
_resp = requests.get(_url, timeout=30)
_lines = _resp.text.strip().split('\n')

def _val(v):
    """Extrai valor numérico do formato MongoDB Extended JSON."""
    if isinstance(v, dict):
        if "$numberInt" in v:    return int(v["$numberInt"])
        if "$numberDouble" in v: return float(v["$numberDouble"])
        if "$numberLong" in v:   return int(v["$numberLong"])
        return None
    return v

rows = []
for line in _lines:
    doc = json.loads(line)
    start_coords = doc.get("start station location", {}).get("coordinates", [None, None])
    end_coords   = doc.get("end station location", {}).get("coordinates", [None, None])
    rows.append({
        "tripduration": _val(doc.get("tripduration")),
        "birth year": _val(doc.get("birth year")),
        "usertype": doc.get("usertype"),
        "start station longitude": _val(start_coords[0]) if start_coords[0] else None,
        "start station latitude":  _val(start_coords[1]) if start_coords[1] else None,
        "end station longitude":   _val(end_coords[0]) if end_coords[0] else None,
        "end station latitude":    _val(end_coords[1]) if end_coords[1] else None,
    })

df_raw = pd.DataFrame(rows)
print(f"Registros baixados: {len(df_raw)}")

df_raw['dist'] = np.sqrt(
    (df_raw['end station latitude']  - df_raw['start station latitude'])**2 +
    (df_raw['end station longitude'] - df_raw['start station longitude'])**2
)
df_raw['birth year'] = pd.to_numeric(df_raw['birth year'], errors='coerce')
df_raw['age'] = 2026 - df_raw['birth year']

df = df_raw[['tripduration', 'dist', 'age', 'usertype']].copy()
df = df[
    (df['tripduration'].between(60, 10800)) &
    (df['dist'] > 0.001) &
    (df['age'].between(15, 80))
].dropna(subset=['tripduration', 'dist', 'age'])

print(f"Registros disponíveis após filtro: {len(df)}")

-----------------

## EXERCÍCIO 1

Antes de construir qualquer modelo, **analise a correlação** entre as features disponíveis e o target `tripduration`.

Tarefas:
- Calcule o coeficiente de Pearson entre `dist` e `tripduration`, e entre `age` e `tripduration`.
- Plote um gráfico de dispersão para cada par (com 1 000 pontos amostrados).
- Com base nos resultados, justifique qual feature deve ser usada para prever a duração.

In [0]:
# --- EXERCÍCIO 1: Análise de Correlação ---

r_dist, p_dist = stats.pearsonr(df['dist'], df['tripduration'])
r_age,  p_age  = stats.pearsonr(df['age'], df['tripduration'])

print("=== Correlação de Pearson ===")
print(f"dist vs tripduration: r = {r_dist:.4f} (p = {p_dist:.2e})")
print(f"age  vs tripduration: r = {r_age:.4f} (p = {p_age:.2e})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample = df.sample(min(1000, len(df)), random_state=42)

axes[0].scatter(sample['dist'], sample['tripduration'], alpha=0.4, s=15)
axes[0].set_xlabel('dist')
axes[0].set_ylabel('tripduration')
axes[0].set_title(f'dist vs tripduration (r={r_dist:.4f})')

axes[1].scatter(sample['age'], sample['tripduration'], alpha=0.4, s=15, color='orange')
axes[1].set_xlabel('age')
axes[1].set_ylabel('tripduration')
axes[1].set_title(f'age vs tripduration (r={r_age:.4f})')

plt.tight_layout()
plt.show()

print(f"\nConclusão: dist (r={r_dist:.4f}) tem correlação {'mais forte' if abs(r_dist)>abs(r_age) else 'mais fraca'} que age (r={r_age:.4f}) com tripduration.")
print("Portanto, 'dist' deve ser a feature principal para prever a duração da viagem.")

-----------------

## EXERCÍCIO 2

Construa um modelo de **Regressão Linear Simples** para prever `tripduration` a partir de `dist`.

Tarefas:
- Aplique o pipeline completo: `train_test_split` → `fit` → `predict` → métricas.
- Imprima o coeficiente β₁ e interprete: *o que ele diz sobre a relação entre distância e duração?*
- Plote o gráfico de dispersão com a linha de regressão.

In [0]:
# --- EXERCÍCIO 2: Regressão Linear Simples ---

X = df[['dist']].values
y = df['tripduration'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test, y_pred_test)

print("=== Regressão Linear Simples (dist → tripduration) ===")
print(f"β₀ (intercepto): {model.intercept_:.4f}")
print(f"β₁ (coeficiente dist): {model.coef_[0]:.4f}")
print(f"RMSE treino: {rmse_train:.4f}")
print(f"RMSE teste:  {rmse_test:.4f}")
print(f"R² treino:   {r2_train:.4f}")
print(f"R² teste:    {r2_test:.4f}")
print(f"\nInterpretação: a cada aumento de 1 unidade em dist, a duração aumenta ~{model.coef_[0]:.2f} segundos.")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_test, y_test, alpha=0.3, s=10, label='Dados reais')
x_line = np.linspace(X_test.min(), X_test.max(), 100).reshape(-1, 1)
ax.plot(x_line, model.predict(x_line), color='red', linewidth=2, label='Reta de regressão')
ax.set_xlabel('dist')
ax.set_ylabel('tripduration')
ax.set_title('Regressão Linear Simples: dist vs tripduration')
ax.legend()
plt.tight_layout()
plt.show()

-----------------

## EXERCÍCIO 3

Adicione `age` como segunda feature e avalie se o modelo melhora.

Tarefas:
- Construa um modelo de **Regressão Linear Múltipla** com `dist` e `age`.
- Compare o R² e o RMSE com os do Exercício 2.
- **Diagnóstico de overfitting**: treine modelos com `PolynomialFeatures` de grau 1 a 10 (feature: apenas `dist`).  
  Plote a curva de RMSE treino vs teste por grau e identifique em qual grau o overfitting começa.

In [0]:
# --- EXERCÍCIO 3: Regressão Linear Múltipla + Diagnóstico de Overfitting ---

# Parte 1: Regressão múltipla com dist e age
X_multi = df[['dist', 'age']].values
y_multi = df['tripduration'].values

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

model_multi = LinearRegression()
model_multi.fit(X_train_m, y_train_m)

y_pred_train_m = model_multi.predict(X_train_m)
y_pred_test_m  = model_multi.predict(X_test_m)

rmse_test_m  = np.sqrt(mean_squared_error(y_test_m, y_pred_test_m))
r2_test_m    = r2_score(y_test_m, y_pred_test_m)

print("=== Regressão Linear Múltipla (dist + age → tripduration) ===")
print(f"β₀: {model_multi.intercept_:.4f}")
print(f"β₁ (dist): {model_multi.coef_[0]:.4f}")
print(f"β₂ (age):  {model_multi.coef_[1]:.4f}")
print(f"RMSE teste: {rmse_test_m:.4f}")
print(f"R² teste:   {r2_test_m:.4f}")

print(f"\n--- Comparação com Exercício 2 ---")
print(f"Regressão Simples:  RMSE teste = {rmse_test:.4f}, R² = {r2_test:.4f}")
print(f"Regressão Múltipla: RMSE teste = {rmse_test_m:.4f}, R² = {r2_test_m:.4f}")

# Parte 2: Diagnóstico de overfitting com PolynomialFeatures (grau 1 a 10, feature: dist)
X_poly = df[['dist']].values
y_poly = df['tripduration'].values

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_poly, y_poly, test_size=0.2, random_state=42)

degrees = range(1, 11)
rmse_train_list, rmse_test_list = [], []

for d in degrees:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d)),
        ('lin', LinearRegression())
    ])
    pipe.fit(X_train_p, y_train_p)
    rmse_train_list.append(np.sqrt(mean_squared_error(y_train_p, pipe.predict(X_train_p))))
    rmse_test_list.append(np.sqrt(mean_squared_error(y_test_p, pipe.predict(X_test_p))))

print("\n=== RMSE por Grau Polinomial (feature: dist) ===")
for d, rt, re in zip(degrees, rmse_train_list, rmse_test_list):
    print(f"Grau {d:2d}: RMSE treino = {rt:.4f}, RMSE teste = {re:.4f}, Gap = {re-rt:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(degrees, rmse_train_list, 'o-', label='RMSE Treino')
ax.plot(degrees, rmse_test_list, 's-', label='RMSE Teste')
ax.set_xlabel('Grau do Polinômio')
ax.set_ylabel('RMSE')
ax.set_title('Overfitting: RMSE Treino vs Teste por Grau')
ax.legend()
ax.set_xticks(degrees)
plt.tight_layout()
plt.show()

# Identificar onde o overfitting começa
# Estratégia: encontrar o grau com menor RMSE de teste; a partir de onde o teste piora,
# o modelo começa a sofrer overfitting
best_degree = list(degrees)[np.argmin(rmse_test_list)]
overfit_degree = None
for i in range(list(degrees).index(best_degree) + 1, len(degrees)):
    if rmse_test_list[i] > rmse_test_list[i - 1]:
        overfit_degree = list(degrees)[i]
        break

print(f"\nMelhor grau (menor RMSE teste): {best_degree}")
if overfit_degree:
    print(f"Overfitting começa no grau: {overfit_degree} (RMSE teste começa a piorar)")
else:
    print("Não há overfitting significativo até o grau 10 — o modelo estabiliza.")
print("Observação: a relação entre dist e tripduration é majoritariamente linear;")
print("polinômios de grau alto não trazem ganho e aumentam o risco de overfitting.")

-----------------

## EXERCÍCIO 4

Compare **Regressão Linear**, **Lasso** e **Ridge** no modelo com `dist` e `age` como features.

Tarefas:
- Teste Lasso e Ridge com `alpha` ∈ {0.1, 1, 10, 100, 1000}.
- Monte uma tabela com RMSE Teste e R² por modelo/alpha.
- Identifique a melhor combinação.
- Plote os coeficientes β dos três modelos (alpha = 100) lado a lado e comente o que o Lasso faz diferente.

In [0]:
# --- EXERCÍCIO 4: Comparação Linear, Lasso, Ridge ---

X_cmp = df[['dist', 'age']].values
y_cmp = df['tripduration'].values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cmp, y_cmp, test_size=0.2, random_state=42)

alphas = [0.1, 1, 10, 100, 1000]
results = []

# Linear Regression (sem alpha)
lr = LinearRegression()
lr.fit(X_train_c, y_train_c)
y_pred = lr.predict(X_test_c)
results.append({
    'Modelo': 'Linear',
    'Alpha': '-',
    'RMSE Teste': np.sqrt(mean_squared_error(y_test_c, y_pred)),
    'R²': r2_score(y_test_c, y_pred)
})

# Lasso
for a in alphas:
    m = Lasso(alpha=a, max_iter=10000)
    m.fit(X_train_c, y_train_c)
    y_pred = m.predict(X_test_c)
    results.append({
        'Modelo': 'Lasso',
        'Alpha': a,
        'RMSE Teste': np.sqrt(mean_squared_error(y_test_c, y_pred)),
        'R²': r2_score(y_test_c, y_pred)
    })

# Ridge
for a in alphas:
    m = Ridge(alpha=a)
    m.fit(X_train_c, y_train_c)
    y_pred = m.predict(X_test_c)
    results.append({
        'Modelo': 'Ridge',
        'Alpha': a,
        'RMSE Teste': np.sqrt(mean_squared_error(y_test_c, y_pred)),
        'R²': r2_score(y_test_c, y_pred)
    })

df_results = pd.DataFrame(results)
print("=== Comparação de Modelos ===")
print(df_results.to_string(index=False))

# Melhor combinação
best = df_results.loc[df_results['R²'].idxmax()]
print(f"\nMelhor combinação: {best['Modelo']} (alpha={best['Alpha']})")
print(f"  R² = {best['R²']:.4f}, RMSE = {best['RMSE Teste']:.4f}")

# Plot dos coeficientes (alpha=100)
lr_coefs    = LinearRegression().fit(X_train_c, y_train_c).coef_
lasso_coefs = Lasso(alpha=100, max_iter=10000).fit(X_train_c, y_train_c).coef_
ridge_coefs = Ridge(alpha=100).fit(X_train_c, y_train_c).coef_

fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(2)
width = 0.25
ax.bar(x_pos - width, lr_coefs, width, label='Linear')
ax.bar(x_pos, lasso_coefs, width, label='Lasso (α=100)')
ax.bar(x_pos + width, ridge_coefs, width, label='Ridge (α=100)')
ax.set_xticks(x_pos)
ax.set_xticklabels(['β₁ (dist)', 'β₂ (age)'])
ax.set_ylabel('Valor do Coeficiente')
ax.set_title('Coeficientes dos Modelos (α=100)')
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nComentário: O Lasso tende a zerar coeficientes menos relevantes (regularização L1),")
print("enquanto o Ridge reduz proporcionalmente mas não zera (regularização L2).")

-----------------